# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madaraf/Starter-Notebooks-Assignment-Flyrank-/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb huggingface_hub scikit-learn pandas

In [ ]:

import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Base paths
REL = 'hf://datasets/FlyRank/internship-warehouse'

Paste your Hugging Face READ token (hf_...): ··········


In [ ]:

# Feature month (March 2026)
MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Outcome month (April 2026)
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

# Dimension table
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

In [ ]:
features_df = con.sql(f"""
    WITH page_month AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions)                        AS impressions_month,
            SUM(gsc_clicks)                             AS clicks_month,
            AVG(gsc_avg_position)                       AS avg_position_month,
            COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions
        FROM {MONTH}
        GROUP BY content_hash_id, client_hash_id
        HAVING SUM(gsc_impressions) >= 100
    )
    SELECT
        p.content_hash_id,
        p.client_hash_id,
        p.impressions_month,
        p.clicks_month,
        ROUND(p.clicks_month * 1.0 / NULLIF(p.impressions_month, 0), 4) AS ctr_month,
        p.avg_position_month,
        p.days_with_impressions,
        DATE_DIFF('day', d.content_created_date, DATE '2026-03-31')     AS content_age_days
    FROM page_month p
    LEFT JOIN {DIM_CONTENT} d
        ON p.content_hash_id = d.content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# Target window aggregation
april_impressions = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS impressions_april
    FROM {APRIL}
    GROUP BY content_hash_id
""").df()

# Merge and calculate label
labeled_df = features_df.merge(april_impressions, on="content_hash_id", how="inner")
labeled_df["pct_change"] = (
    (labeled_df["impressions_april"] - labeled_df["impressions_month"])
    / labeled_df["impressions_month"]
)
labeled_df["is_declining_forward_label"] = (labeled_df["pct_change"] <= -0.20).astype(int)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

TARGET = "is_declining_forward_label"
honest_features = ["impressions_month", "ctr_month", "avg_position_month",
                   "days_with_impressions", "content_age_days"]

# Automatic guard: label ingredients must never be features
assert "pct_change" not in honest_features and "impressions_april" not in honest_features

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



I am reading the paper the way I would want my own work read: respectfully, and asking how each claim could be made even stronger. The paper is written for a broad audience and already lists several limitations, so my questions are about the next level of rigor, not about errors.

Finding 1 (from the paper): Finding 4, "Refresh timing" — pages 365+ days old refreshed within 30 days show 3.2x health (10.7 to 34.5) and 57x impressions (71 to 4,039).

Where does the comparison come from? As I read it, this compares two groups of old pages, refreshed and not refreshed. If pages were chosen for refresh because they already looked promising, part of the gap could come from the selection, not from the refresh itself.
Is the outcome independent of the change? Health score includes impressions, so the 3.2x health figure and the 57x impressions figure are partly the same signal. A measure the score does not contain (for example clicks) would add independent evidence.
What would make it stronger? A before/after view of the same pages, or a comparison with similar pages that were not refreshed, reported with group sizes. Until then, I would describe the result as an observed association and decision-support for choosing refresh candidates, not a measured causal effect.

Finding 2 (from the paper): Finding 2, "The Performance Curve" — content peaks at 61-90 days, dips at 271-365 days, and rebounds at 365+.

Does the validation design support the claim? Each age bucket contains different pages measured at one moment, so the chart compares cohorts, not the life of the same pages. A curve like this can also come from which pages are still active in each bucket.
Possible survivor effect. The analysis uses active pages only (impressions > 0 and sessions > 0), so old pages that stopped earning impressions are not in the older buckets. The paper flags survivor bias for one cell in Finding 8; the same idea may apply to the 365+ rebound. The paper also notes that this rebound is not evidence that age reverses decline on its own, which fits this concern.
What would make it stronger? Following the same pages over time (a cohort view), and reporting how many pages leave each bucket.
Link to my own work: my Week 4 signal audit found a non-monotonic staleness pattern (decline rate peaked at 91-180 days, then fell), which is directionally similar. It uses a different measure and a single month, so I treat it as consistent with this finding, not confirmation of it.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(y_true)[order[:k]].mean()

# --- Baseline rule (Week 4), now fitted on TRAIN rows only ---
def position_bucket(pos):
    if pos <= 3: return "top_3"
    elif pos <= 10: return "page_1"
    elif pos <= 20: return "page_2"
    return "beyond_page_2"

def fit_baseline(train_df):
    pb = train_df["avg_position_month"].apply(position_bucket)
    expected = train_df.groupby(pb)["ctr_month"].mean().to_dict()
    gap = (pb.map(expected) - train_df["ctr_month"]).clip(lower=0)
    return {"expected": expected,
            "gap_max": gap.max() or 1.0,
            "age_max": train_df["content_age_days"].fillna(0).max() or 1.0}

def score_baseline(df, p):
    pb = df["avg_position_month"].apply(position_bucket)
    gap = (pb.map(p["expected"]) - df["ctr_month"]).clip(lower=0).fillna(0)
    age = df["content_age_days"].fillna(0)
    return (0.6 * gap / p["gap_max"] + 0.4 * age / p["age_max"]).values

def evaluate(train_df, test_df, feats):
    Xtr, ytr = train_df[feats].fillna(0), train_df[TARGET]
    Xte, yte = test_df[feats].fillna(0), test_df[TARGET].values
    scores = {"baseline_rule": score_baseline(test_df, fit_baseline(train_df))}
    tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(Xtr, ytr)
    scores["decision_tree"] = tree.predict_proba(Xte)[:, 1]
    rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced_subsample",
                                random_state=42, n_jobs=-1).fit(Xtr, ytr)
    scores["random_forest"] = rf.predict_proba(Xte)[:, 1]
    rows = []
    for name, s in scores.items():
        p50 = precision_at_k(yte, s, 50)
        rows.append({"method": name, "precision_at_50": p50, "base_rate": yte.mean(),
                     "lift_vs_base_rate": p50 / yte.mean(), "roc_auc": roc_auc_score(yte, s)})
    return pd.DataFrame(rows)

In [ ]:
# BEFORE: naive random-row split
tr, te = train_test_split(labeled_df, test_size=0.25, random_state=42, stratify=labeled_df[TARGET])
random_res = evaluate(tr, te, honest_features)
print("BEFORE — random row split")
print(random_res.round(3))

BEFORE — random row split
          method  precision_at_50  base_rate  lift_vs_base_rate  roc_auc
0  baseline_rule             0.80      0.518              1.545    0.607
1  decision_tree             0.98      0.518              1.892    0.675
2  random_forest             0.92      0.518              1.776    0.713


In [ ]:
# AFTER: 5-fold split grouped by client
folds = []
for i, (tr_i, te_i) in enumerate(GroupKFold(n_splits=5).split(labeled_df, groups=labeled_df["client_hash_id"])):
    r = evaluate(labeled_df.iloc[tr_i], labeled_df.iloc[te_i], honest_features)
    r["fold"] = i
    r["test_clients"] = labeled_df.iloc[te_i]["client_hash_id"].nunique()
    r["test_rows"] = len(te_i)
    folds.append(r)
grouped = pd.concat(folds)

print("Per-fold Precision@50 (watch how much it moves):")
print(grouped.pivot(index="fold", columns="method", values="precision_at_50").round(3))
print("\nPer-fold base rate, clients, rows:")
print(grouped.groupby("fold")[["base_rate", "test_clients", "test_rows"]].first().round(3))

print("\nAFTER — grouped mean ± std across folds:")
print(grouped.groupby("method")[["precision_at_50", "lift_vs_base_rate", "roc_auc"]].agg(["mean", "std"]).round(3))

Per-fold Precision@50 (watch how much it moves):
method  baseline_rule  decision_tree  random_forest
fold                                               
0                0.52           0.16           0.48
1                1.00           1.00           0.90
2                0.92           0.90           0.74
3                0.64           0.34           0.24
4                0.94           0.66           0.90

Per-fold base rate, clients, rows:
      base_rate  test_clients  test_rows
fold                                    
0         0.431             1      21633
1         0.739             4      20007
2         0.529            13      19934
3         0.377            14      19934
4         0.519            12      19933

AFTER — grouped mean ± std across folds:
              precision_at_50        lift_vs_base_rate        roc_auc       
                         mean    std              mean    std    mean    std
method                                                              

In [ ]:
# The before/after table for the notebook
before = random_res.set_index("method")[["precision_at_50", "roc_auc"]].add_suffix("_random")
after = grouped.groupby("method")[["precision_at_50", "roc_auc"]].mean().add_suffix("_grouped")
ba = before.join(after)
ba["auc_gap (random - grouped)"] = ba["roc_auc_random"] - ba["roc_auc_grouped"]
ba.round(3)

,precision_at_50_random,roc_auc_random,precision_at_50_grouped,roc_auc_grouped,auc_gap (random - grouped)
method,,,,,
baseline_rule,0.80,0.607,0.804,0.614,-0.007
decision_tree,0.98,0.675,0.612,0.586,0.089
random_forest,0.92,0.713,0.652,0.621,0.092


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Paste this markdown first:[cite: 1]

**Timeline.** Features use March 2026 only (`content_age_days` is anchored to 2026-03-31). The label uses April 2026 only. The windows do not overlap.[cite: 1]

| Check | Result |
| :--- | :--- |
| Label-derived features | `pct_change`, `impressions_april` excluded, enforced by an assert[cite: 1] |
| Product flags or existing-system scores | none in the feature set[cite: 1] |
| IDs as features | none; `client_hash_id` used only for grouping[cite: 1] |
| Split grouped by the repeating entity | yes (Section 2)[cite: 1] |
| Base rate printed next to every metric | yes (Section 2)[cite: 1] |

Then run the checks that need real numbers.[cite: 1]




**Does any single feature carry suspiciously all of the skill?**

In [ ]:
def cv_auc(feats, n_splits=3):
    aucs = []
    for tr_i, te_i in GroupKFold(n_splits=n_splits).split(labeled_df, groups=labeled_df["client_hash_id"]):
        a, b = labeled_df.iloc[tr_i], labeled_df.iloc[te_i]
        m = RandomForestClassifier(n_estimators=60, max_depth=6, class_weight="balanced_subsample",
                                   random_state=42, n_jobs=-1).fit(a[feats].fillna(0), a[TARGET])
        aucs.append(roc_auc_score(b[TARGET], m.predict_proba(b[feats].fillna(0))[:, 1]))
    return float(np.mean(aucs))

full_auc = cv_auc(honest_features)
rows = [{"feature_dropped": "(none)", "auc": full_auc, "change": 0.0}]
for f in honest_features:
    a = cv_auc([c for c in honest_features if c != f])
    rows.append({"feature_dropped": f, "auc": a, "change": a - full_auc})
pd.DataFrame(rows).round(3)

,feature_dropped,auc,change
0,(none),0.644,0.000
1,impressions_month,0.641,-0.003
2,ctr_month,0.606,-0.038
3,avg_position_month,0.638,-0.006
4,days_with_impressions,0.657,0.013
5,content_age_days,0.643,-0.001


label-adjacent feature. pct_change = (April − March) / March, so impressions_month is the denominator of your label. It is knowable in March, so it is not future leakage, but it is mechanically tied to the label: pages with an unusually high March tend to fall back (regression to the mean).

**population selection**. The skill says to check whether the rows you keep depend on the outcome window. Your inner join with April keeps only pages that have April rows.

In [ ]:
dropped = features_df[~features_df["content_hash_id"].isin(labeled_df["content_hash_id"])]
print(f"March pages (>=100 impressions): {len(features_df):,}")
print(f"Kept after joining April:        {len(labeled_df):,}")
print(f"Dropped (no April rows):         {len(dropped):,}  ({len(dropped)/len(features_df):.1%})")
print("\nMarch impressions of the dropped pages:")
print(dropped["impressions_month"].describe().round(1))
print(f"\nOverall label base rate in kept pages: {labeled_df[TARGET].mean():.3f}")

March pages (>=100 impressions): 101,441
Kept after joining April:        101,441
Dropped (no April rows):         0  (0.0%)

March impressions of the dropped pages:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: impressions_month, dtype: float64

Overall label base rate in kept pages: 0.518


In [ ]:
tr_i, te_i = next(iter(GroupKFold(n_splits=5).split(labeled_df, groups=labeled_df["client_hash_id"])))
tr_df, te_df = labeled_df.iloc[tr_i], labeled_df.iloc[te_i].copy()
rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced_subsample",
                            random_state=42, n_jobs=-1).fit(tr_df[honest_features].fillna(0), tr_df[TARGET])
te_df["rf_score"] = rf.predict_proba(te_df[honest_features].fillna(0))[:, 1]
te_df["page"] = te_df["content_hash_id"].astype(str).str[:10]
cols = ["page", "impressions_month", "ctr_month", "avg_position_month", "content_age_days", "pct_change", "rf_score"]

print("CONFIDENT FALSE ALARMS (model said decline, page did not decline):")
print(te_df[te_df[TARGET] == 0].sort_values("rf_score", ascending=False).head(5)[cols].round(3))
print("\nMISSED DECLINES (page declined, model gave a low score):")
print(te_df[te_df[TARGET] == 1].sort_values("rf_score").head(5)[cols].round(3))

CONFIDENT FALSE ALARMS (model said decline, page did not decline):
             page  impressions_month  ctr_month  avg_position_month  \
94685  content_01              486.0        0.0               7.300   
94985  content_ee              763.0        0.0              10.382   
94938  content_9f              514.0        0.0              10.409   
48236  content_94              621.0        0.0              10.510   
44256  content_c2              468.0        0.0              10.503   

       content_age_days  pct_change  rf_score  
94685               410      -0.126     0.869  
94985               412       0.160     0.868  
94938               412       0.161     0.868  
48236               412       0.401     0.868  
44256               412      -0.197     0.868  

MISSED DECLINES (page declined, model gave a low score):
              page  impressions_month  ctr_month  avg_position_month  \
93824   content_82             8183.0      0.006               4.190   
93823   content_

In [15]:
top50 = te_df.sort_values("rf_score", ascending=False).head(50)
print(top50["client_hash_id"].nunique(), "clients;", top50["content_age_days"].nunique(), "distinct ages")
print("label rate in top 50:", top50[TARGET].mean().round(3))

1 clients; 2 distinct ages
label rate in top 50: 0.48


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

I re-read the claims I made in Week 5 and compared each one with the evidence from Sections 2 and 3. Some claims held, one did not, and all of them needed more careful wording.

| My original claim (Week 5) | Why it goes further than the evidence | Rewritten claim |
| :--- | :--- | :--- |
| "Random Forest beats the baseline (0.640 vs 0.500), about 28% better." | It came from one split with 11 test clients. The baseline's expected-CTR table was built using test rows. No base rate was shown. Under five client-grouped folds the result reversed in direction, and the spread is large. | "On five client-grouped folds, the Random Forest's Precision@50 averaged 0.652 ± 0.287, versus 0.804 ± 0.211 for the rule (fold base rates 0.38 to 0.74; overall 0.518). The gap is smaller than the fold-to-fold spread, so this is **observed** on one March→April window and **directional**: I cannot name a winner, and my Week 5 result did not hold under this design. On AUC the two are close (0.621 vs 0.614)." |
| "The model leans on CTR, matching my Week 4 signals." | Permutation importance came from one split, and it describes this model, not the world. | "`ctr_month` was the most-used input in this model (**measured** two ways: Week 5 permutation importance, and a Week 6 drop-one test where removing it lowered AUC from 0.644 to 0.606, the largest drop of any feature). This is consistent with my Week 4 bucket table. It is not proof of what drives declines." |
| "The model under-flags very old pages." | It came from one split with small age buckets, and I did not re-test it in this audit. | "In one held-out set, very old pages declined far more often than the model predicted (**observed**). I have not re-checked this across the five grouped folds, so it needs re-checking on other clients before I rely on it." |
| "This model helps editors find declining pages." | I did not test real editor use, and the label is a proxy. On five client-grouped folds the model is not clearly better than the simple rule. | "The ranked list is **decision-support**: a starting point for human review of pages whose impressions fell 20%+ from March to April. It does not show that a refresh will help, and it does not show that the model beats the simple rule." |
| "The model's mistakes are mostly random." (implicit in Week 5) | Section 3 shows patterns in the errors. | "In the examples I inspected (**observed**, small sample): confident false alarms were old pages (about 410 days) with CTR of 0, several with near-identical scores, which may point to a batch of similar pages (**hypothesis**, not tested). Missed declines were young pages (14 to 42 days) with high impressions and good positions, possibly launch-period spikes that faded (**hypothesis**, not tested)." |

### What I would say now

- **Random split flattered the learned models.** Random-split AUC was 0.675 (tree) and 0.713 (forest) versus 0.586 and 0.621 when grouped by client, a gap of about 0.09. The rule showed no gap (0.607 vs 0.614).
- **Results depend heavily on which clients are held out.** Per-fold Precision@50 for the rule ranged from 0.52 to 1.00, and one fold contained a single client with 21,633 rows.
- **No leakage signature was found.** No single feature carried the model, and the full-feature AUC was 0.644.

### Limitations I am naming

- One month pair (March features, April label), and one random seed.
- The label is a proxy: a fall of 20% or more in impressions. Pages with no April impressions count as declining, and a page that fell 19.7% counts as not declining.
- The base rate is high (0.518), so Precision@50 must always be read next to it.
- Precision@50 judges only 50 pages per fold (about 0.25% of rows) and is noisy. AUC is the steadier comparison.
- Fold sizes are uneven: one fold has 1 client and another has 14.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.